# c04 — Inferência local vs remota para o analista de fraude

**Competency #4 — Private inference, with a justified local-vs-cloud comparison.**

## The question

Where should System 2's generation LLM run for the Vigil Private-RAG fraud analyst?
**[ADR-003](../docs/adr/ADR-003-inference-strategy.md)** decided the answer: **LOCAL via GPT4All**
(Meta-Llama-3.1-8B-Instruct, Q4_0 quant, RTX 3070, n_ctx=4096). Cloud is **comparison-only**
via OpenAI `gpt-5.4-nano` against synthetic corpus cases. This notebook produces the
evidence behind that decision: privacy posture, latency, cost, valid-JSON rate, and
two qualitative grounding spot-checks.

## Hard rules in play

- **HR-3 — privacy.** The shipped (local) path keeps case data on-box. The cloud
  comparison sends **only synthetic corpus cases** (`corpus/cases/*.md`) — they are
  masked/synthetic by construction; no real cardholder data exists.
- **No keys in code.** `OPENAI_API_KEY` is read from the environment only.
  If absent, the notebook still runs end-to-end with the local half and a clean
  skip notice on cloud cells.
- **LAT-1 does not apply.** System 2 is async/advisory (ADR-001). We measure
  latency for the comparison; we do not gate on it.
- **No notebook code in `src/`. No retrieval here** — the probe prompt holds the
  case body raw. Retrieval lands in c05; this notebook deliberately captures the
  *no-retrieval baseline* so c05's with-vs-without comparison has a paired anchor.


## 1. Setup — imports, cases, environment posture


In [1]:
import os
import sys
import statistics
from pathlib import Path

project_root = next(
    p for p in (Path.cwd(), Path.cwd().parent) if (p / 'corpus').exists()
)
for path in (project_root, project_root / 'src'):
    s = str(path)
    if s not in sys.path:
        sys.path.insert(0, s)

import pandas as pd

from vigil.generation.generate import generate
from vigil.generation.schema import Disposition
from vigil.generation.json_repair import extract_json, repair_json, InvalidDispositionError
from vigil.generation.local import (
    LOCAL_MODEL_NAME,
    LOCAL_N_CTX,
    get_local_device,
    get_gpu_failure_reason,
)
from vigil.generation.probe import PROBE_PROMPT_TEMPLATE, build_prompt
from vigil.generation.cloud import CLOUD_MODEL_NAME, CLOUD_PRICE_PROMPT_PER_MILLION, CLOUD_PRICE_COMPLETION_PER_MILLION

CASES_DIR = project_root / 'corpus' / 'cases'
CASE_PATHS = sorted(CASES_DIR.glob('case-*.md'))
CASES = [(p.name, p.read_text(encoding='utf-8')) for p in CASE_PATHS]

HAVE_OPENAI = bool(os.environ.get('OPENAI_API_KEY'))

print(f'loaded {len(CASES)} synthetic cases from {CASES_DIR}')
print(f'local model:  {LOCAL_MODEL_NAME} (n_ctx={LOCAL_N_CTX})')
print(f'cloud model:  {CLOUD_MODEL_NAME}')
print(f'OPENAI_API_KEY present: {HAVE_OPENAI}')
if not HAVE_OPENAI:
    print('  -> cloud cells will skip cleanly; local-only results below.')


loaded 10 synthetic cases from C:\Users\user\Desktop\vigil\corpus\cases
local model:  Meta-Llama-3.1-8B-Instruct-128k-Q4_0.gguf (n_ctx=4096)
cloud model:  gpt-5.4-nano
OPENAI_API_KEY present: True


## 2. The probe prompt — frozen, byte-identical across engines

Intentionally minimal: role + task + format. **Not** optimized — real prompt
engineering is c02. This prompt exists only to hold the prompt constant so the
comparison measures the **engine**, not the prompt.


In [2]:
# PROBE_PROMPT_TEMPLATE and build_prompt are imported from vigil.generation.probe.
# Regression guard for the original brace bug: tests/generation/test_probe.py.
print(PROBE_PROMPT_TEMPLATE.replace('{case_body}', '<case markdown inlined here>'))


You are a fraud-case analyst. Read the Case below and produce a Disposition.

Return ONLY a JSON object with these fields, no prose, no markdown:
{
  "recommendation": one of "allow" | "block" | "review-continue",
  "confidence":    one of "low" | "medium" | "high",
  "reason_codes":  list of short snake_case strings (required when recommendation is "block"),
  "cited_sources": non-empty list of corpus paths you relied on (e.g. "typologies/card-testing.md"),
  "rationale":     one short paragraph explaining the recommendation
}

--- CASE ---
<case markdown inlined here>
--- END CASE ---


## 3. Smoke tests — both engines, BEFORE the loops

These run **once** before the 30-call benchmark. If either fails, the benchmark
cells must not be entered. Honest reporting: if cloud is unavailable, the notebook
still runs the local half — the comparison table just has empty cloud columns and
the privacy/cost analysis still leans on the local result.


### 3a. Local — GPT4All load + one generation (self-validating)

First run will download the GGUF (~4 GB) to `~/.cache/gpt4all/`. Subsequent runs use
the cached weights. The loader tries the GPU backend (Vulkan via GPT4All's
`device='gpu'`) first and runs a tiny probe generation **immediately after load** —
because the CUDA backend on this host loaded successfully but emitted nothing
(silent failure). If the probe returns empty/whitespace, the GPU model is
discarded and the loader reloads on CPU. The smoke cell below additionally
asserts non-empty output: a stalled or silently-empty engine fails loudly here,
not 30 calls into the benchmark.


In [3]:
smoke = generate('Reply with the single word OK.', backend='local')
assert smoke.text and smoke.text.strip(), (
    f'local smoke produced EMPTY output — engine is broken. '
    f'device={get_local_device()!r}, gpu_failure_reason={get_gpu_failure_reason()!r}'
)
print(f'backend:      {smoke.backend}')
print(f'device:       {get_local_device()}')
print(f'model:        {smoke.model_name}')
print(f'latency_ms:   {smoke.latency_ms:,.1f}')
print(f'output (truncated): {smoke.text[:200]!r}')
gpu_reason = get_gpu_failure_reason()
if gpu_reason:
    print()
    print(f'NOTE: GPU path was rejected -> {gpu_reason}')
    print('Local benchmark below runs on CPU. See §11 for the control finding.')


[local] loaded Meta-Llama-3.1-8B-Instruct-128k-Q4_0.gguf on GPU (n_ctx=4096), probe ok: 'Say yes.\nI am a 30'


backend:      local
device:       gpu
model:        Meta-Llama-3.1-8B-Instruct-128k-Q4_0.gguf
latency_ms:   9,535.5
output (truncated): 'OK.'


### 3b. Cloud — GET /v1/models, confirm gpt-5.4-nano

Skips cleanly if the key isn't in the environment. When present, lists the
available models and aborts with a clear message if `gpt-5.4-nano` isn't there.


In [4]:
if not HAVE_OPENAI:
    print('OPENAI_API_KEY not set — skipping cloud smoke test.')
    print('Cloud cells below will skip identically; local-only comparison stands.')
else:
    from vigil.generation.cloud import list_available_models
    available = list_available_models()
    print(f'OpenAI /v1/models returned {len(available)} ids')
    nano_present = CLOUD_MODEL_NAME in available
    print(f'"{CLOUD_MODEL_NAME}" present: {nano_present}')
    if not nano_present:
        like_nano = [m for m in available if 'nano' in m.lower()]
        raise RuntimeError(
            f'gpt-5.4-nano not in API model list. Nano-class models present: {like_nano}. '
            f'Update CLOUD_MODEL_NAME in src/vigil/generation/cloud.py and re-run.'
        )


OpenAI /v1/models returned 118 ids
"gpt-5.4-nano" present: True


## 4. Benchmark harness

10 cases × 3 runs per engine = 30 calls per side. We record per-call latency,
valid-JSON rate (after extract, after one repair pass), and (cloud only) token
counts and cost. The qualitative grounding pass on a sample reuses the same rows.


In [5]:
N_RUNS = 1

def validate_disposition(text: str) -> tuple[bool, bool, dict | None, str | None]:
    '''Return (valid_after_extract, valid_after_repair, parsed_dict, error).'''
    valid_extract = False
    valid_repair = False
    parsed: dict | None = None
    err: str | None = None
    try:
        parsed = extract_json(text)
        Disposition.model_validate(parsed)
        valid_extract = True
        valid_repair = True
        return valid_extract, valid_repair, parsed, None
    except (InvalidDispositionError, Exception) as exc:
        err = f'extract: {exc!r}'
    try:
        parsed = repair_json(text)
        Disposition.model_validate(parsed)
        valid_repair = True
        return valid_extract, valid_repair, parsed, None
    except (InvalidDispositionError, Exception) as exc:
        err = (err or '') + f' | repair: {exc!r}'
    return valid_extract, valid_repair, parsed, err

def run_benchmark(backend: str, cases, n_runs: int) -> list[dict]:
    rows = []
    for case_name, case_body in cases:
        prompt = build_prompt(case_body)
        for run_idx in range(n_runs):
            result = generate(prompt, backend=backend)
            ok_extract, ok_repair, parsed, err = validate_disposition(result.text)
            # Live progress so a stalled call is visible immediately.
            print(
                f"[{backend}] {case_name} run {run_idx+1}/{n_runs}: "
                f"{result.latency_ms:.0f}ms valid={ok_repair}",
                flush=True,
            )
            rows.append({
                'backend': result.backend,
                'model': result.model_name,
                'case': case_name,
                'run': run_idx,
                'latency_ms': result.latency_ms,
                'valid_after_extract': ok_extract,
                'valid_after_repair': ok_repair,
                'prompt_tokens': result.prompt_tokens,
                'completion_tokens': result.completion_tokens,
                'cost_usd': result.cost_usd,
                'output_text': result.text,
                'parsed': parsed,
                'error': err,
            })
    return rows


## 5. Local benchmark — GPT4All over all 10 cases × 3 runs


In [6]:
local_rows = run_benchmark('local', CASES, N_RUNS)
local_df = pd.DataFrame(local_rows)
print(f'local: {len(local_df)} calls')
local_df[['case', 'run', 'latency_ms', 'valid_after_extract', 'valid_after_repair']].head(10)


[local] case-account-takeover-shipping-change.md run 1/1: 77553ms valid=True
[local] case-bin-attack-blocked.md run 1/1: 74256ms valid=True
[local] case-clean-fraud-released-then-cb.md run 1/1: 68536ms valid=True
[local] case-cnp-velocity-burst.md run 1/1: 76355ms valid=True
[local] case-friendly-fraud-chargeback.md run 1/1: 63392ms valid=True
[local] case-high-value-allowed-3ds.md run 1/1: 70341ms valid=True
[local] case-phishing-card-test.md run 1/1: 79248ms valid=True
[local] case-promo-abuse-multi-account.md run 1/1: 69054ms valid=True
[local] case-refund-fraud-pattern.md run 1/1: 69609ms valid=True
[local] case-triangulation-marketplace.md run 1/1: 78418ms valid=True
local: 10 calls


,case,run,latency_ms,valid_after_extract,valid_after_repair
0,case-account-takeover-shipping-change.md,0,77553.0107,True,True
1,case-bin-attack-blocked.md,0,74255.5928,True,True
2,case-clean-fraud-released-then-cb.md,0,68535.9599,True,True
3,case-cnp-velocity-burst.md,0,76355.1522,True,True
4,case-friendly-fraud-chargeback.md,0,63392.2139,True,True
5,case-high-value-allowed-3ds.md,0,70340.5674,True,True
6,case-phishing-card-test.md,0,79248.1597,True,True
7,case-promo-abuse-multi-account.md,0,69053.8057,True,True
8,case-refund-fraud-pattern.md,0,69608.7230,True,True
9,case-triangulation-marketplace.md,0,78418.2674,True,True


## 6. Cloud benchmark — OpenAI `gpt-5.4-nano` (skipped if no key)


In [7]:
if not HAVE_OPENAI:
    print('OPENAI_API_KEY not set — skipping cloud benchmark.')
    cloud_df = pd.DataFrame()
else:
    cloud_rows = run_benchmark('cloud', CASES, N_RUNS)
    cloud_df = pd.DataFrame(cloud_rows)
    # GPT-5 family rejects non-default temperature with HTTP 400. cloud.py
    # detects this on the first call and falls back to default sampling for
    # the rest. Surface the note here so the table is read correctly.
    from vigil.generation.cloud import temperature_was_dropped
    if temperature_was_dropped():
        print()
        print('NOTE: gpt-5.4-nano rejected the requested temperature; cloud calls used '
              'default sampling. Comparison is still valid — same prompt, same model, '
              'same seed posture across cases.')
    print(f'cloud: {len(cloud_df)} calls')
    display_cols = ['case', 'run', 'latency_ms', 'valid_after_extract', 'prompt_tokens', 'completion_tokens', 'cost_usd']
    cloud_df[display_cols].head(10)


[cloud] case-account-takeover-shipping-change.md run 1/1: 3659ms valid=True
[cloud] case-bin-attack-blocked.md run 1/1: 2548ms valid=True
[cloud] case-clean-fraud-released-then-cb.md run 1/1: 1884ms valid=True
[cloud] case-cnp-velocity-burst.md run 1/1: 1947ms valid=True
[cloud] case-friendly-fraud-chargeback.md run 1/1: 2489ms valid=True
[cloud] case-high-value-allowed-3ds.md run 1/1: 1934ms valid=True
[cloud] case-phishing-card-test.md run 1/1: 2663ms valid=True
[cloud] case-promo-abuse-multi-account.md run 1/1: 2320ms valid=True
[cloud] case-refund-fraud-pattern.md run 1/1: 1762ms valid=True
[cloud] case-triangulation-marketplace.md run 1/1: 1936ms valid=True
cloud: 10 calls


## 7. Comparison table — privacy / latency / valid-JSON / cost

The table the rubric (#4) asks for: same prompt, same cases, two engines,
measured numbers.


In [8]:
def summarize(df: pd.DataFrame, engine: str) -> dict:
    if df.empty:
        return {
            'engine': engine,
            'calls': 0,
            'latency_mean_ms': None,
            'latency_p50_ms': None,
            'latency_p95_ms': None,
            'valid_extract_rate': None,
            'valid_repair_rate': None,
            'mean_cost_usd': None,
            'total_cost_usd': None,
        }
    latencies = df['latency_ms'].tolist()
    return {
        'engine': engine,
        'calls': len(df),
        'latency_mean_ms': round(statistics.mean(latencies), 1),
        'latency_p50_ms': round(statistics.median(latencies), 1),
        'latency_p95_ms': round(sorted(latencies)[int(0.95 * (len(latencies) - 1))], 1),
        'valid_extract_rate': round(df['valid_after_extract'].mean(), 3),
        'valid_repair_rate': round(df['valid_after_repair'].mean(), 3),
        'mean_cost_usd': round(df['cost_usd'].dropna().mean(), 6) if df['cost_usd'].notna().any() else 0.0,
        'total_cost_usd': round(df['cost_usd'].dropna().sum(), 6) if df['cost_usd'].notna().any() else 0.0,
    }

# Label the local row with the device that actually produced the output
# (loader auto-falls back if the GPU probe was empty — see §11).
_device = get_local_device() or 'not-loaded'
_gpu_note = '' if (_device == 'gpu' or get_gpu_failure_reason() is None) else ' [GPU fallback]'
local_label = f'local ({LOCAL_MODEL_NAME}) [{_device}]{_gpu_note}'

summary_rows = [
    summarize(local_df, local_label),
    summarize(cloud_df, f'cloud ({CLOUD_MODEL_NAME})'),
]
summary_df = pd.DataFrame(summary_rows)
summary_df


,engine,calls,latency_mean_ms,latency_p50_ms,latency_p95_ms,valid_extract_rate,valid_repair_rate,mean_cost_usd,total_cost_usd
0,local (Meta-Llama-3.1-8B-Instruct-128k-Q4_0.gg...,10,72676.1,72298.1,78418.3,1.0,1.0,0.000000,0.000000
1,cloud (gpt-5.4-nano),10,2314.3,2133.5,2663.4,1.0,1.0,0.000125,0.001253


In [9]:
# Axes the rubric explicitly names
axes = pd.DataFrame([
    {'axis': 'Privacy — does case data leave the host?', 'local': 'No', 'cloud': 'Yes (sent to OpenAI)'},
    {'axis': 'Versioning / offline availability',         'local': 'Yes (GGUF file)', 'cloud': 'No (vendor API)'},
    {'axis': 'Per-call cost',                              'local': '$0.00 after one-time download', 'cloud': f'~${CLOUD_PRICE_PROMPT_PER_MILLION}/M prompt, ${CLOUD_PRICE_COMPLETION_PER_MILLION}/M completion'},
    {'axis': 'Latency control',                            'local': 'Bound by local GPU/CPU', 'cloud': 'Bound by network + vendor queue'},
    {'axis': 'Reproducibility for the grader',             'local': 'Key-free, one download', 'cloud': 'Requires OPENAI_API_KEY'},
])
axes


,axis,local,cloud
0,Privacy — does case data leave the host?,No,Yes (sent to OpenAI)
1,Versioning / offline availability,Yes (GGUF file),No (vendor API)
2,Per-call cost,$0.00 after one-time download,"~$0.05/M prompt, $0.4/M completion"
3,Latency control,Bound by local GPU/CPU,Bound by network + vendor queue
4,Reproducibility for the grader,"Key-free, one download",Requires OPENAI_API_KEY


## 8. Qualitative grounding — two cases, side-by-side

Two cases hand-picked for **category coverage** (one card-testing burst, one
account-takeover) — not for outcome. For each: does the LLM cite corpus paths
that actually exist on disk?

**Caveat — path-existence is a weak grounding signal.** Both engines score
~100% citation-existence here *without any retrieval*, because the corpus
follows a guessable convention (`typologies/<typology>.md`,
`reason_codes/<code>.md`, `glossary.md#<term>`) and a frontier model can
recall plausible paths from training data alone. A path that exists on disk
is not the same as a path whose *content actually supports the claim* in the
rationale — and that distinction is what c05 needs to grade. So:

**Framing for c05.** This notebook captures the deliberate **no-retrieval
baseline** for the c05 with-vs-without comparison. The right metric for c05 is
not path-existence (already saturated) but **faithfulness**: given the
retrieved chunk, does the rationale rest on text actually present in that
chunk? That's the hallucination-reduction evidence c05 owes — a fact about the
*content* the citations point to, not just whether the file exists.


In [10]:
GROUNDING_CASES = ['case-cnp-velocity-burst.md', 'case-account-takeover-shipping-change.md']

def citation_existence_rate(parsed: dict | None, project_root: Path) -> tuple[int, int]:
    if not parsed or not isinstance(parsed, dict):
        return 0, 0
    sources = parsed.get('cited_sources') or []
    if not isinstance(sources, list):
        return 0, 0
    corpus = project_root / 'corpus'
    found = 0
    for raw in sources:
        if not isinstance(raw, str):
            continue
        path_part = raw.split('#', 1)[0]
        if (corpus / path_part).exists():
            found += 1
    return found, len(sources)

def first_row(df, case_name):
    if df.empty:
        return None
    matched = df[df['case'] == case_name]
    return matched.iloc[0] if not matched.empty else None

for case_name in GROUNDING_CASES:
    print('=' * 78)
    print(case_name)
    print('=' * 78)
    for engine_df, label in [(local_df, 'local'), (cloud_df, 'cloud')]:
        row = first_row(engine_df, case_name)
        if row is None:
            print(f'-- {label}: SKIPPED (no rows)')
            continue
        parsed = row['parsed']
        found, total = citation_existence_rate(parsed, project_root)
        print(f'-- {label} ({row["model"]}) lat={row["latency_ms"]:.0f}ms valid={row["valid_after_repair"]}')
        if parsed:
            print(f'   recommendation: {parsed.get("recommendation")!r}  confidence: {parsed.get("confidence")!r}')
            print(f'   cited_sources ({found}/{total} exist on disk): {parsed.get("cited_sources")}')
            print(f'   reason_codes: {parsed.get("reason_codes")}')
            print(f'   rationale: {(parsed.get("rationale") or "")[:300]}')
        else:
            print(f'   raw output (truncated): {row["output_text"][:300]!r}')
    print()


case-cnp-velocity-burst.md
-- local (Meta-Llama-3.1-8B-Instruct-128k-Q4_0.gguf) lat=76355ms valid=True
   recommendation: 'block'  confidence: 'high'
   cited_sources (3/3 exist on disk): ['typologies/card-testing.md', 'reason_codes/visa-10-4-card-absent.md', 'glossary.md#velocity_high']
   reason_codes: ['velocity_high', 'bin_diversity_high']
   rationale: The signals match the card-testing typology cleanly — low-amount burst, high BIN diversity from a tight IP and device cluster, decline-then-success pattern, and prior confirmed-fraud on the same `device_fingerprint`. Authentication did not shift liability. The expected loss from releasing exceeds th
-- cloud (gpt-5.4-nano) lat=1947ms valid=True
   recommendation: 'block'  confidence: 'high'
   cited_sources (3/3 exist on disk): ['typologies/card-testing.md', 'reason_codes/visa-10-4-card-absent.md', 'glossary.md#velocity_high']
   reason_codes: ['velocity_high', 'bin_diversity_high', 'low_amount_anomaly', 'geo_mismatch']
   rationale

## 9. Privacy / cost / latency / control — analysis

### Privacy — the spine of the decision
Vigil's thesis (ADR-001) is **Private RAG**: the data stays on-box. Local embeddings
(ADR-002, MiniLM via sentence-transformers) plus cloud generation would leak the
case + retrieved context to a third party — incoherent with the thesis. Local generation
preserves the posture end-to-end. On the cloud half of this benchmark, we only sent
**synthetic** corpus cases (HR-3-safe by construction). The production posture is
what's graded, and the production posture is local.

### Cost
The local engine is $0/call after the one-time ~4 GB GGUF download. Cloud cost per
call is small at nano-tier prices, but it scales linearly with case volume.
At Vigil's pilot scale (~1k transactions/day, of which only the `review`-queue
subset reaches System 2), the absolute cost is low either way; the *posture*
argument carries more weight than the *spend* argument.

### Latency
Latency is **explicitly off LAT-1** for System 2 (ADR-001). The measured numbers
above are for the comparison's sake, not a fitness function. Local is expected
to be slower per token than cloud at this model size; that's the documented
trade-off and it does not affect the decision (System 2 is async/advisory).

### Control
- The local model is **versioned** (GGUF file hash) and **offline-capable**.
- The cloud model can change under the same id between calls; deprecation
  windows are vendor-controlled. ADR-003's choice of `gpt-5.4-nano` already
  reflects one such shift — GPT-4.1 was on the docs page during ADR drafting
  and is not on the current `developers.openai.com/api/docs/models` page.


## 10. Justification — ship LOCAL

Restated against the measurements above:

| Decision driver | What the numbers show | Verdict |
|---|---|---|
| **Privacy (HR-3)** | Local: case data never leaves the host. Cloud: every call is a third-party round-trip. | **Local** — only coherent option with the Private-RAG thesis (ADR-001). |
| **Reproducibility for the grader** | Local: key-free. Cloud: requires `OPENAI_API_KEY`. | **Local** — the grader runs the system with no spend, no signup. |
| **Cost** | Local: $0/call after download. Cloud: ~cents at nano tier. | **Local** by a wide margin at any scale. |
| **Quality (valid JSON)** | §7 summary table: both engines hit **100% valid_extract = 100% valid_repair** across all 10 cases, with the same `block` / `high`-confidence dispositions and real corpus citations on the spot-checked pair. Local **matches** cloud — no measured gap on this probe. | **Local** — no quality cost paid. The privacy + cost wins come for free at the c04 probe difficulty. |
| **Latency** | Local slower per call; off LAT-1 for System 2 anyway. | **Local — acceptable** because System 2 is async/advisory. |
| **Control / versioning** | Local: GGUF hash pinned. Cloud: vendor-mutable. | **Local** — auditable model identity. |

**Ship local. Keep cloud comparison documented and re-runnable** — that's the
ADR-003 decision, now backed by measurement: shipping local is a privacy + cost
win at **zero measured quality cost** against `gpt-5.4-nano` on this probe.
Harder inputs (retrieved-context prompts in c05, adversarial cases) may still
show a gap; c02's prompt-hardening keeps a head-room buffer either way.


## 11. Limits & risks

- **Local strict-JSON reliability — anticipated risk that did NOT materialize.**
  ADR-003 named small-model JSON output as the load-bearing cost of going local,
  on the basis that quantized 7-8B instruct models historically fumble strict
  JSON. On this probe it didn't show: local hit **100% valid extracts** across
  all 10 cases — no repair pass needed, parity with cloud. The risk is real for
  harder inputs (longer retrieved contexts in c05, adversarial prompts, JSON
  schemas with deeper nesting), so c02's tight role/context/format prompt +
  schema-validated repair retry remains the head-room buffer. Worth noting:
  the c04 probe is short, structured, and free of adversarial framing — that
  fits the easy end of the distribution, and the result generalizes only as
  far as the probe.
- **GPU memory ceiling.** RTX 3070 has 8 GB VRAM. We deliberately cap `n_ctx=4096`
  (not the model's max 128k) and CPU-fallback if VRAM is contested. The fallback
  is fine — System 2 is off LAT-1 — but the notebook prints which path was taken.
- **Silent-empty GPU backend — the actual c04 finding.** On this host (RTX 3070,
  Windows), GPT4All's CUDA backend **loaded successfully and emitted nothing**.
  No exception, no traceback — the SDK returned an empty string. A naive
  "GPU-or-CPU" loader that only catches load exceptions would have happily run
  the entire 10-case benchmark against a non-functional engine and reported a
  bogus 0%-valid-JSON local result. `src/vigil/generation/local.py` now treats
  GPU load as **provisional**: load → tiny probe generation → discard the
  candidate if the probe is empty/whitespace → reload on CPU. The smoke cell in
  §3a additionally asserts non-empty output as a belt-and-suspenders guard. The
  §7 engine label and the §12 footer report the device that **actually produced
  the benchmark output**, not the one the SDK said it loaded. Vulkan
  (GPT4All's auto-GPU path with `device='gpu'`) is the GPU backend tried first
  here; CUDA was previously attempted via the `gpt4all[cuda]` extra and is the
  variant observed to fail silently. For competency #4, this is the **control
  and reliability** axis with evidence: vendor accelerator backends can fail
  in ways the loader doesn't surface, and the only honest defense in a
  high-stakes domain is to validate the output path, not just the load path.
- **GPT-5 family sampling constraints.** `gpt-5.4-nano` rejects non-default
  `temperature` with HTTP 400 and uses `max_completion_tokens` (not
  `max_tokens`). `cloud.py` detects the temperature rejection on the first call
  and silently retries with default sampling for the rest of the benchmark; the
  cloud cell prints a one-line note so the table is read correctly. Same prompt,
  same model, same seed posture across cases — the comparison stays clean.
- **Pricing snapshot.** The cloud cost estimate uses pricing constants in
  `src/vigil/generation/cloud.py`. Verify against OpenAI's pricing page before
  reporting; the constants are deliberately exposed in the import section above.
- **Model deprecation.** `gpt-5.4-nano` is the comparison target as of this run.
  The §3b smoke test will abort with a clear message if it disappears from the
  API listing; pick a current nano-tier replacement and update `CLOUD_MODEL_NAME`.
- **No retrieval here.** This is the deliberate no-retrieval baseline — c05's RAG
  pipeline will improve grounding (cited-source existence rate, faithfulness) on
  the same probe.


## 12. Reproducibility footer


In [11]:
import platform
import gpt4all
import openai
import pydantic

print('Python                :', platform.python_version())
print('Platform              :', platform.platform())
print('gpt4all SDK           :', gpt4all.__version__ if hasattr(gpt4all, '__version__') else 'unknown')
print('openai SDK            :', openai.__version__)
print('pydantic              :', pydantic.VERSION)
print('LOCAL_MODEL_NAME      :', LOCAL_MODEL_NAME)
print('LOCAL_N_CTX           :', LOCAL_N_CTX)
print('CLOUD_MODEL_NAME      :', CLOUD_MODEL_NAME)
print('CLOUD_PRICE_PROMPT    : $%.4f / 1M prompt tokens' % CLOUD_PRICE_PROMPT_PER_MILLION)
print('CLOUD_PRICE_COMPLETION: $%.4f / 1M completion tokens' % CLOUD_PRICE_COMPLETION_PER_MILLION)
print('OPENAI_API_KEY present:', HAVE_OPENAI)
print('local device used     :', get_local_device())
_reason = get_gpu_failure_reason()
if _reason:
    print('gpu rejected because  :', _reason[:200])
print('cases benchmarked     :', len(CASES))
print('runs per case         :', N_RUNS)


Python                : 3.13.12
Platform              : Windows-11-10.0.26200-SP0
gpt4all SDK           : unknown
openai SDK            : 1.109.1
pydantic              : 2.13.4
LOCAL_MODEL_NAME      : Meta-Llama-3.1-8B-Instruct-128k-Q4_0.gguf
LOCAL_N_CTX           : 4096
CLOUD_MODEL_NAME      : gpt-5.4-nano
CLOUD_PRICE_PROMPT    : $0.0500 / 1M prompt tokens
CLOUD_PRICE_COMPLETION: $0.4000 / 1M completion tokens
OPENAI_API_KEY present: True
local device used     : gpu
cases benchmarked     : 10
runs per case         : 1
